# 2교시 실습 — 타이타닉, 누가 살아남았나? 🕵️🚢

> **사용법**
> - 🔵 **수업** = 강사와 함께 실행하는 셀 · 🟡 **셀프** = 집에서 직접 채울 셀(빈칸 `____`)
> - 이 파일은 **연습용(`_blank`)** 입니다. 막히면 정답본(`_solution`)과 대조하세요.
> - 정답본에는 **실행 결과 + 💡해석**이 셀마다 달려 있습니다.

---

## 📭 오늘의 미션

여러분은 오늘 막 **데이터 분석가**로 첫 출근을 했습니다.
팀장님이 책상에 파일 하나를 툭 던지며 말합니다.

> *"이거 1912년 타이타닉호 승객 명단이야. 누가 살아남았는지 데이터에 단서가 있을 거야.*
> *데이터랑 인사부터 하고 → 정리하고 → 누가 살았는지 분석해서 → 그림까지 그려서 보고해줘. 오늘 안에!"*

오늘 우리는 분석가의 **하루 전체**를 따라갑니다:

```
 ① 인사       ② 정제        ③ 분석          ④ 시각화      ⑤ 해석
첫인상 보기 → 빈칸 채우기 → 그룹별 생존율 →  그림으로 →  "누가, 왜"
```

겁먹지 마세요. 한 칸씩 천천히 가면 됩니다. 출발! 🚀

## 🎯 학습 목표 — 이 실습을 끝내면

- 데이터의 **첫인상**을 5가지로 파악할 수 있다 (shape·head·info·describe·value_counts)
- **결측치(빈 칸)** 를 찾아내고 중앙값으로 채울 수 있다
- **groupby**로 그룹별(등급·성별) 생존율을 비교할 수 있다
- **막대그래프·박스플롯**으로 결과를 그림으로 표현할 수 있다
- 숫자 뒤의 **'누가, 왜 살아남았나'를 한 문장으로 해석**할 수 있다

## ① 인사 (1) — 워밍업: 노트북 다루기  🔵

노트북은 **셀(cell)** 단위로 코드를 실행합니다.
셀을 클릭하고 **Shift + Enter** 를 누르면 실행되고, 결과가 바로 아래에 나옵니다. 몸풀기부터.

In [ ]:
print("첫 출근 완료!")   # 따옴표 안 글자를 화면에 그대로 출력
1 + 1                    # 마지막 줄의 계산 결과(2)는 자동으로 표시됨

🤔 **확인**: 실행하면 무엇이 출력되나요? `print(...)`와 마지막 줄 `1+1`의 출력이 어떻게 다른지 보세요.

데이터 분석의 필수 도구 **판다스(pandas)** 와 예제 데이터 도구 **seaborn** 을 불러옵니다.
`import ... as ...` 는 "이 도구를 이런 별명으로 쓸게" 라는 뜻이에요.  🔵

In [ ]:
import pandas as pd      # 표(데이터)를 다루는 도구, 별명 pd
import seaborn as sns      # 예제 데이터 + 시각화 도구
import matplotlib.pyplot as plt   # 그래프 그리는 도구

# --- 그래프 한글 깨짐 방지: 아래는 '그대로 실행'만 하세요 (코랩에서 폰트 자동 설치) ---
import os, urllib.request                       # 파일 확인·다운로드용 기본 도구
from matplotlib import font_manager as fm       # 폰트 등록 담당 도구
_FONT_URL = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"   # 한글 폰트 내려받을 주소
def set_korean_font():                          # 한글 폰트를 켜주는 함수 정의
    cands = ["../data/BMDOHYEON.ttf", "data/BMDOHYEON.ttf", "BMDOHYEON.ttf"]   # 폰트가 있을 만한 후보 위치들
    if not any(os.path.exists(p) for p in cands):   # 후보 중 하나도 없으면
        try: urllib.request.urlretrieve(_FONT_URL, "BMDOHYEON.ttf")   # 인터넷에서 폰트 다운로드 시도
        except Exception: pass                  # 실패해도 그냥 넘어감
    for p in cands:                             # 후보 위치를 하나씩 확인
        if os.path.exists(p):                   # 그 위치에 폰트 파일이 실제로 있으면
            try:
                fm.fontManager.addfont(p)       # 폰트를 matplotlib에 등록
                plt.rcParams["font.family"] = fm.FontProperties(fname=p).get_name()   # 기본 글꼴로 지정
                plt.rcParams["axes.unicode_minus"] = False   # 마이너스(-) 기호 깨짐 방지
                return                          # 성공했으니 함수 종료
            except Exception: pass
    print("(안내) 한글 폰트를 못 찾았어요. 영문 라벨 차트는 정상입니다.")   # 모두 실패 시 안내
set_korean_font()                               # 위에서 만든 함수를 실제로 실행

print("pandas 버전:", pd.__version__)            # 판다스 버전 출력 = 준비 완료 확인

🤔 **확인**: 버전 숫자가 출력되면 성공입니다. (코랩·아나콘다엔 이미 깔려 있어요.)

## ① 인사 (2) — 타이타닉과 첫인사  🔵

데이터를 불러옵니다. seaborn 내장 데이터라 **다운로드 없이** 바로 옵니다.

In [ ]:
df = sns.load_dataset("titanic")   # 타이타닉 예제 데이터를 불러와 df라는 상자에 담기
print("불러오기 완료! 변수 이름은 df 입니다.")   # 안내 문구 출력

🤔 **확인**: `df` 에 데이터가 담겼습니다. 다음 셀부터 이 `df` 를 하나씩 뜯어봅니다.

### `head()` — 일단 눈으로 보기  🔵
백 마디 설명보다 직접 보는 게 빠릅니다. 맨 위 5줄만.

In [ ]:
df.head()   # 표의 맨 위 5줄만 미리 보기

🤔 **직접 해석**: 어떤 컬럼들이 보이나요? 그중 '생존'과 관련 있어 보이는 변수 2~3개를 골라 적어보세요.

### `shape` — 규모 파악  🟡
몇 명이 탔고(행), 정보는 몇 가지(열)일까요?
> 💡 힌트: `df.shape` → `(행, 열)`

In [ ]:
# TODO: 데이터의 (행 수, 열 수)
df.____

🤔 **직접 해석**: 출력된 두 숫자는 각각 무엇을 뜻하나요? (앞=___, 뒤=___)

### `info()` — 신상명세서 + 빈 칸 찾기  🔵
각 컬럼의 **타입**과 **빈 칸(결측치)** 을 한눈에. 분석가가 가장 먼저 확인하는 것 중 하나예요.

In [ ]:
df.info()   # 각 컬럼의 타입과 빈 칸(결측치) 개수를 한눈에

🤔 **직접 해석**: `age` 와 `deck` 의 Non-Null 개수를 전체(891)와 비교해보세요. 어느 컬럼에 빈 칸이 많나요?

### `describe()` — 숫자 요약  🟡
숫자 컬럼들의 평균·최소·최대·중앙값(50%)을 한 번에.
> 💡 힌트: `df.describe()`

In [ ]:
# TODO: 숫자 컬럼 요약 통계
df.____()

🤔 **직접 해석**: `age` 평균과 `fare` 의 평균·최댓값(max)을 보세요. 요금은 평균과 최댓값 차이가 큰가요? (1교시 '평균의 함정')

### `value_counts()` — 범주 세기  🟡
범주형 변수는 "각 값이 몇 번 나오나"를 셉니다. 성별과 객실등급을 세어보세요.
> 💡 힌트: `df["sex"].value_counts()`

In [ ]:
# TODO: 성별 인원 수
df["sex"].____()

In [ ]:
# TODO: 객실 등급별 인원 수
df["class"].____()

🤔 **직접 해석**: 남성과 여성 중 누가 더 많나요? 어느 등급이 가장 많나요? 한 문장으로 정리해보세요.

## ① 인사 (3) — 궁금한 곳 콕 집어 보기

큰 그림을 봤으니 이제 **특정 부분**을 자세히. 분석가의 기본 손놀림 몇 가지를 익힙니다.

### 평균 구하기  🟡
평균 나이와 평균 요금을 구해봅시다.
> 💡 힌트: `df["age"].mean()`

In [ ]:
# TODO: 평균 나이와 평균 요금
print("평균 나이:", round(df["____"].____(), 1))
print("평균 요금:", round(df["fare"].mean(), 1))

🤔 **직접 해석**: 평균 나이는 몇 세인가요? 요금의 '평균'은 믿을 만한가요, 극단값에 휘둘렸을까요?

### 정렬해서 보기 — `sort_values()`  🟡
요금이 가장 비쌌던 승객 5명은? 요금 높은 순으로.
> 💡 힌트: `df.sort_values("fare", ascending=False)`

In [ ]:
# TODO: 요금(fare) 높은 순 상위 5명
df.sort_values("____", ascending=False).____()

🤔 **직접 해석**: 가장 비싼 요금 승객들의 등급(`pclass`)과 생존(`survived`)은? 공통점이 보이나요?

### 조건으로 걸러내기 — 필터링  🔵
"18세 미만 어린이는 몇 명?" 조건을 만족하는 행만 골라냅니다.

In [ ]:
children = df[df["age"] < 18]   # 나이가 18 미만인 행(어린이)만 골라 따로 담기
print("18세 미만 어린이 수:", len(children), "명")   # 골라낸 행 개수 = 어린이 수

🤔 **직접 해석**: 18세 미만 어린이는 몇 명인가요? 전체 891명 중 많은 편인가요?

## ② 정제 (Clean) — 나이의 빈 칸 채우기  🟡

`info()`에서 봤듯 `age`에 **177명의 빈 칸**이 있었죠. 나이는 우리 분석의 핵심이라 버릴 수 없으니 **채워야** 합니다.
극단값이 많지 않은 나이는 **중앙값(median)** 으로 채우는 게 무난합니다. 채운 뒤 결측이 **0이 됐는지** 확인하세요.
> 💡 힌트: `df["age"].fillna(df["age"].median())`  *(자세한 정제 기법은 3교시에서!)*

In [ ]:
# TODO: age 결측치를 '중앙값(median)'으로 채우고 0이 됐는지 확인
before = df["age"].isnull().sum()
df["age"] = df["age"].fillna(df["age"].____())
after = df["age"].isnull().sum()
print(f"나이 결측치: {before}개 → {after}개")
print("채우는 데 쓴 중앙값:", df["age"].median(), "세")

🤔 **직접 해석**: 결측이 0이 됐나요? 채운 중앙값은 몇 세인가요? 왜 평균 대신 중앙값으로 채웠을까요?

## ③ 분석 (Group) — 누가 더 살아남았나  🔵

분석의 심장. **`groupby`** 로 "그룹별 생존율"을 구합니다.
`survived`가 1/0이라 평균(`mean`)을 내면 그게 곧 **생존율**입니다(0.63 = 63% 생존).

먼저 **객실 등급(pclass)** 별 생존율.

In [ ]:
df.groupby("pclass")["survived"].mean().round(3)   # 등급별로 묶어 생존(1/0) 평균=생존율, 소수 3자리

🤔 **확인**: 1등석과 3등석의 생존율 차이는 얼마나 큰가요? 등급이 낮아질수록 생존율은?

### 직접 분석 — 성별 생존율  🟡
**성별(`sex`)** 로 생존율을 직접 구해보세요. "여성과 어린이 먼저"가 사실이었을까요?
> 💡 힌트: `df.groupby("sex")["survived"].mean()`

In [ ]:
# TODO: 성별(sex) 생존율
df.groupby("____")["survived"].mean().round(3)

🤔 **직접 해석**: 여성과 남성의 생존율 차이는? 등급 차이(63%↔24%)와 성별 차이 중 어느 쪽이 더 극적인가요?

### 두 조건을 겹치면? — 등급 × 성별  🔵
진짜 분석가는 여기서 멈추지 않습니다. **"등급과 성별을 동시에"** 보면 더 깊은 진실이 나옵니다.

In [ ]:
df.groupby(["sex", "pclass"])["survived"].mean().round(3)   # 성별 × 등급 두 조건으로 동시에 묶어 생존율 계산

🤔 **확인**: 3등석 여성과 1등석 남성 중 누가 더 살아남았나요? 이 결과는 '등급'과 '성별' 중 무엇이 더 중요했음을 뜻하나요?

## ④ 시각화 (Visualize) — 숫자를 그림으로  🔵

표의 숫자는 회의에서 설득력이 약합니다. 등급별 생존율을 **막대그래프**로 바꿉니다.
*(차트는 4교시에서 본격적으로 배웁니다 — 오늘은 맛보기!)*

In [ ]:
sns.barplot(data=df, x="pclass", y="survived")   # x=등급, y=생존율 막대그래프 그리기
plt.title("객실 등급별 생존율")   # 그래프 제목 달기
plt.xlabel("객실 등급"); plt.ylabel("생존율")   # 가로축·세로축 이름 달기
plt.show()   # 완성된 그래프를 화면에 표시

🤔 **확인**: 막대가 등급에 따라 어떤 모양(올라감/내려감)인가요? 표 숫자보다 그림이 더 와닿나요?

### 직접 시각화 ① — 성별 × 등급 한 그림에  🟡
③에서 발견한 "성별이 더 중요"를 한 장으로. 막대에 `hue="sex"` 를 더하면 등급별로 남/녀 막대가 나란히.
> 💡 힌트: `sns.barplot(data=df, x="pclass", y="survived", hue="sex")`

In [ ]:
# TODO: 등급별 생존율을 성별(hue)로 나눠 막대그래프로
sns.barplot(data=df, x="pclass", y="survived", hue="____")
plt.title("등급 x 성별 생존율")
plt.xlabel("객실 등급"); plt.ylabel("생존율")
plt.show()

🤔 **직접 해석**: 같은 등급 안에서 항상 여성이 더 높나요? 3등석 여성 막대와 1등석 남성 막대 중 어느 게 더 높은가요?

### 직접 시각화 ② — 생존/사망별 나이 분포  🟡
산 사람과 죽은 사람의 **나이 분포**가 다를까요? 박스플롯으로 비교하세요.
> 💡 힌트: `sns.boxplot(data=df, x="survived", y="age")`

In [ ]:
# TODO: 생존 여부(survived)별 나이(age) 분포를 박스플롯으로
sns.boxplot(data=df, x="____", y="age")
plt.title("생존 여부별 나이 분포 (0=사망, 1=생존)")
plt.xlabel("생존 여부"); plt.ylabel("나이")
plt.show()

🤔 **직접 해석**: 두 상자(사망/생존)의 가운데 선(중앙값) 높이가 많이 다른가요? 나이가 성별·등급만큼 강하게 가른 것 같나요?

## ⑤ 해석 (Interpret) — "그래서 무엇을 알았나"  🟡

**가장 중요한 단계.** 코드는 숫자를 줄 뿐, *결론을 내는 건 사람*입니다.
아래 셀은 핵심 숫자를 한 번에 출력합니다. 빈칸을 채워 실행한 뒤, `[해석]` 에 **내 결론을 한 문단** 적어보세요.

In [ ]:
# TODO: 빈칸을 채워 핵심 숫자를 출력하고, 맨 아래 [해석]을 한 문단으로 직접 적어보세요
c1 = df.groupby("pclass")["survived"].mean()
c2 = df.groupby("____")["survived"].mean()
print("=== 타이타닉 생존 분석 요약 ===")
print(f"전체 생존율      : {df['survived'].mean():.0%}")
print(f"1등석 vs 3등석   : {c1[1]:.0%}  vs  {c1[3]:.0%}")
print(f"여성 vs 남성     : {c2['female']:.0%}  vs  {c2['male']:.0%}")
print()
print("[해석] (여기에 내가 알아낸 것을 한 문단으로 적어보세요)")

🤔 **직접 해석**: 출력된 숫자로 '누가 더 살아남았나, 왜인가'를 한 문단으로. 가장 강력한 증거(숫자) 하나를 꼭 포함하세요.

## 🎯 미션 완료 — 팀장님께 보고

오늘 여러분은 분석가의 **하루 전체**를 처음부터 끝까지 직접 해냈습니다:

```
 ① 인사          ② 정제           ③ 분석             ④ 시각화        ⑤ 해석
첫인상 5종 →   나이 중앙값 28 →   등급·성별 생존율 →  막대·박스플롯 →  "성별 > 등급"
              결측 177→0              나눠보기                        한 문단 결론
```

그리고 **데이터가 가설을 뒤집는 순간**(3등석 여성 > 1등석 남성)을 직접 목격했습니다.
이게 분석의 진짜 재미예요 — 미리 정한 답이 아니라, **데이터가 말하게 하는 것.**

| 도구 | 무엇을 알려주나 |
|---|---|
| `head/shape/info/describe` | 데이터 첫인상 + 빈 칸 |
| `value_counts` | 범주별 개수 |
| `fillna` | 결측치 채우기(정제) |
| `groupby(...).mean()` | **그룹별 생존율(분석의 심장)** |
| `barplot / boxplot` | 그림으로 비교 |

> ## "도구는 변해도, 좋은 질문을 던지고 결과를 해석하는 힘은 사람의 몫이다."

> 🏠 **셀프스터디**: 같은 5단계를 **다른 데이터**로 직접 돌려보세요.
> `df = sns.load_dataset("penguins")` 로 바꿔, "종(species)별 몸무게는 어떻게 다를까?"를 `groupby`·`boxplot`으로 분석해보면 똑같이 됩니다. 방법은 늘 같아요! 🚀